# Comparación estadística: Optuna-Best (100 épocas) frente a baseline, v1 y v2

Cálculo de métricas **caso a caso** (con `case_id`) para el modelo `nnUNetTrainerOptunaBest_100epochs` del notebook 22 y comparación mediante Wilcoxon y t-test pareado con `v1_lam02` y `baseline`. Se emplea el mismo método del notebook 19 (mismas funciones de métricas y mismo criterio de significación: `wilcoxon_p < 0.05` y `ttest_p < 0.05`). No se realiza ningún entrenamiento.

Requiere el archivo `comparacion_metricas_completo.csv` en Drive (generado por el notebook 19), que contiene los datos caso a caso de baseline, v1, v2 y el barrido de λ.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!python -m pip install -q SimpleITK medpy scikit-image scipy pandas

In [ ]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

DATASET_FOLDER_NAME = "Dataset501_ImageCAS"
CONFIGURATION = "3d_fullres"
FOLD = 0

GT_FOLDER = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations"

CSV_METRICAS_PREVIAS = BASE / "comparacion_metricas_completo.csv"  # del notebook 19
CSV_SALIDA = BASE / "comparacion_metricas_con_optuna_best.csv"
CSV_TESTS_SALIDA = BASE / "tests_pareados_optuna_best.csv"

# Nombre de la carpeta de resultados del notebook 22
TRAINER_OPTUNA_BEST = "nnUNetTrainerOptunaBest_100epochs"

print("CSV previas existe:", CSV_METRICAS_PREVIAS.exists())
print("GT folder existe:", GT_FOLDER.exists())

CSV previas existe: True
GT folder existe: True


## 1. Funciones de métricas

Idénticas a las del notebook 19, incluidas `hd95_principal` y `hd95_filtrado`, que el notebook 22 no calculaba.


In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

MIN_VOXELS_FILTRO = 20
CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def filtrar_componentes_pequenas(pred, min_voxels=MIN_VOXELS_FILTRO):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return pred
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    etiquetas_validas = set(np.where(tamanos >= min_voxels)[0] + 1)
    mascara = np.isin(labeled, list(etiquetas_validas))
    return mascara.astype(pred.dtype)


def solo_componente_principal(pred):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return pred
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    return (labeled == idx_mayor).astype(pred.dtype)


def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)

    dice = dice_score(pred, gt)
    cldice = hard_cldice(pred, gt)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)

    hd95_crudo = hd95_seguro(pred, gt, spacing)
    hd95_principal = hd95_seguro(solo_componente_principal(pred), gt, spacing)
    hd95_filtrado = hd95_seguro(filtrar_componentes_pequenas(pred), gt, spacing)

    return {
        "dice": dice,
        "cldice": cldice,
        "n_componentes": n_componentes,
        "cobertura_principal": cobertura,
        "hd95_crudo_mm": hd95_crudo,
        "hd95_principal_mm": hd95_principal,
        "hd95_filtrado_mm": hd95_filtrado,
    }

print("Funciones de metricas cargadas (identicas al notebook 19).")

Funciones de metricas cargadas (identicas al notebook 20).


## 2. Métricas caso a caso para Optuna-Best (con `case_id` para el emparejamiento)


In [ ]:
def carpeta_validacion(trainer_name):
    return (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )

carpeta_pred = carpeta_validacion(TRAINER_OPTUNA_BEST)
print("Carpeta de validacion:", carpeta_pred)
print("Existe:", carpeta_pred.exists())

archivos_pred = sorted(carpeta_pred.glob("*.nii.gz"))
print(f"{len(archivos_pred)} casos encontrados")

filas_optuna_best = []
for pred_path in archivos_pred:
    case_id = pred_path.name.replace(".nii.gz", "")
    gt_path = GT_FOLDER / pred_path.name
    if not gt_path.exists():
        print(f"  [{case_id}] sin ground truth, se omite")
        continue
    metricas = calcular_metricas_caso(pred_path, gt_path)
    metricas["modelo"] = "optuna_best"
    metricas["case_id"] = case_id
    filas_optuna_best.append(metricas)

df_optuna_best = pd.DataFrame(filas_optuna_best)
print("\nCasos calculados:", len(df_optuna_best))
df_optuna_best.head()

Carpeta de validacion: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results/Dataset501_ImageCAS/nnUNetTrainerOptunaBest_100epochs__nnUNetPlans__3d_fullres/fold_0/validation
Existe: True
40 casos encontrados

Casos calculados: 40


,dice,cldice,n_componentes,cobertura_principal,hd95_crudo_mm,hd95_principal_mm,hd95_filtrado_mm,modelo,case_id
0,0.708353,0.785847,19,0.455470,46.130755,55.771703,46.133233,optuna_best,ImageCAS_0011
1,0.711029,0.741445,19,0.376815,31.016712,54.323828,30.957749,optuna_best,ImageCAS_0017
2,0.718039,0.785442,17,0.519407,36.189241,67.564665,36.189425,optuna_best,ImageCAS_0018
3,0.604108,0.688889,29,0.515199,62.075325,67.202220,62.076693,optuna_best,ImageCAS_0021
4,0.787866,0.820162,4,0.394146,4.900777,72.670309,4.900777,optuna_best,ImageCAS_0028


## 3. Unión con los datos existentes (baseline, v1, v2 y barrido de λ, notebook 19)


In [ ]:
df_previo = pd.read_csv(CSV_METRICAS_PREVIAS)
print("Modelos ya presentes:", sorted(df_previo['modelo'].unique()))

df_todo = pd.concat([df_previo, df_optuna_best], ignore_index=True)
df_todo = df_todo.drop_duplicates(subset=['modelo', 'case_id'], keep='last')
df_todo.to_csv(CSV_SALIDA, index=False)

print(f"\nTotal filas: {len(df_todo)}")
print(f"Guardado en: {CSV_SALIDA}")

Modelos ya presentes: ['baseline', 'lam01', 'lam04', 'lam08', 'v1_lam02', 'v2_lam02']

Total filas: 280
Guardado en: /content/drive/MyDrive/TFM_nnUNet/comparacion_metricas_con_optuna_best.csv


## 4. Resumen por modelo


In [ ]:
resumen = df_todo.groupby("modelo").agg(
    dice_mean=("dice", "mean"),
    cldice_mean=("cldice", "mean"),
    n_componentes_mean=("n_componentes", "mean"),
    cobertura_mean=("cobertura_principal", "mean"),
    hd95_crudo_mean=("hd95_crudo_mm", "mean"),
    hd95_principal_mean=("hd95_principal_mm", "mean"),
    hd95_filtrado_mean=("hd95_filtrado_mm", "mean"),
).round(4)

resumen

,dice_mean,cldice_mean,n_componentes_mean,cobertura_mean,hd95_crudo_mean,hd95_principal_mean,hd95_filtrado_mean
modelo,,,,,,,
baseline,0.7459,0.8069,17.150,0.4810,24.0244,59.6199,24.0524
lam01,0.7428,0.8016,17.450,0.4833,24.4737,60.4414,24.4616
lam04,0.7537,0.8185,15.125,0.4718,20.1636,61.4079,20.1545
lam08,0.7401,0.8085,15.625,0.4838,23.6521,60.9716,23.6462
optuna_best,0.7502,0.8113,16.800,0.4758,20.7128,60.7968,20.7387
v1_lam02,0.7549,0.8221,14.725,0.4829,21.0636,61.0859,21.0751
v2_lam02,0.7482,0.8094,16.100,0.4798,23.3501,60.6592,23.3644


## 5. Tests pareados: Optuna-Best frente a v1_lam02 y baseline

Mismo criterio que en el notebook 19: Wilcoxon y t-test pareado sobre los mismos 40 casos. La condición `significativo_ambos_p<0.05` exige consenso entre ambos tests.


In [ ]:
from scipy import stats

METRICAS_A_TESTEAR = ["dice", "cldice", "cobertura_principal", "hd95_crudo_mm"]
MODELOS_REFERENCIA = ["baseline", "v1_lam02"]  # contra que se compara optuna_best

pivote = {}
for metrica in METRICAS_A_TESTEAR:
    pivote[metrica] = df_todo.pivot(index="case_id", columns="modelo", values=metrica)

resultados_tests = []
for metrica in METRICAS_A_TESTEAR:
    tabla = pivote[metrica]
    if "optuna_best" not in tabla.columns:
        print(f"[{metrica}] no hay columna optuna_best, se omite")
        continue
    for referencia in MODELOS_REFERENCIA:
        if referencia not in tabla.columns:
            continue
        pareja = tabla[[referencia, "optuna_best"]].dropna()
        if len(pareja) < 5:
            continue
        try:
            w_stat, w_p = stats.wilcoxon(pareja[referencia], pareja["optuna_best"])
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        t_stat, t_p = stats.ttest_rel(pareja[referencia], pareja["optuna_best"])
        resultados_tests.append({
            "metrica": metrica,
            "referencia": referencia,
            "n": len(pareja),
            f"media_{referencia}": pareja[referencia].mean(),
            "media_optuna_best": pareja["optuna_best"].mean(),
            "wilcoxon_p": w_p,
            "ttest_p": t_p,
            "significativo_ambos_p<0.05": (w_p < 0.05) and (t_p < 0.05),
        })

df_tests = pd.DataFrame(resultados_tests).round(4)
df_tests.to_csv(CSV_TESTS_SALIDA, index=False)
print(f"Guardado en: {CSV_TESTS_SALIDA}\n")
df_tests

Guardado en: /content/drive/MyDrive/TFM_nnUNet/tests_pareados_optuna_best.csv



,metrica,referencia,n,media_baseline,media_optuna_best,wilcoxon_p,ttest_p,significativo_ambos_p<0.05,media_v1_lam02
0,dice,baseline,40,0.7459,0.7502,0.0451,0.0233,True,NaN
1,dice,v1_lam02,40,NaN,0.7502,0.0106,0.0190,True,0.7549
2,cldice,baseline,40,0.8069,0.8113,0.1876,0.1022,False,NaN
3,cldice,v1_lam02,40,NaN,0.8113,0.0002,0.0001,True,0.8221
4,cobertura_principal,baseline,40,0.4810,0.4758,0.0200,0.6009,False,NaN
5,cobertura_principal,v1_lam02,40,NaN,0.4758,0.0002,0.0175,True,0.4829
6,hd95_crudo_mm,baseline,40,24.0244,20.7128,0.0007,0.0021,True,NaN
7,hd95_crudo_mm,v1_lam02,40,NaN,20.7128,0.9735,0.6200,False,21.0636


## 6. Interpretación de la tabla de tests

- **`media_baseline` / `media_v1_lam02` frente a `media_optuna_best`**: comparación directa de los promedios.
- **`significativo_ambos_p<0.05` = True** frente a `v1_lam02`: la diferencia (mejora o empeoramiento) de Optuna-Best respecto a v1 es estadísticamente significativa.
- **`significativo_ambos_p<0.05` = False** frente a `v1_lam02`: no hay evidencia suficiente de que Optuna-Best difiera de v1 con estos 40 casos, es decir, ambos son estadísticamente equivalentes aunque las medias difieran ligeramente.

La tabla permite determinar si la búsqueda de Optuna aportó una mejora real, un empate estadístico o un empeoramiento significativo frente al v1 manual.
